# S1 - calibrate + extract (`_final` + `_pooled` + source_overt adjunct)

Target this session at **240-270 min** wall clock; hard boundary **300 min**. Do not start a stage/condition that the calibrated projection says cannot finish (analysis_plan.md §7).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'main'
PINNED_COMMIT = '37c22f4fa0d8a1098017f5518cdeb0b7ad4cf5dd'   # exact commit this session runs against

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, f"wrong commit: {commit} != {PINNED_COMMIT}"
print("checked out", commit)

## 3. Install dependencies, check GPU

In [ ]:
!pip -q install -r requirements.txt
# Colab preinstalls torchao; it breaks transformers on quantized/8-bit
# loads (see 04b). Matches colab_unified_{analysis,training}.ipynb.
!pip uninstall -y torchao || true
!nvidia-smi

## 4. Persistent storage

In [ ]:
import os; os.makedirs('/content/drive/MyDrive/dpo_v2', exist_ok=True)

## 5. Throughput calibration

In [ ]:
!python -m src.analysis.v2_pipeline calibrate --stage M3 --n-prompts 32

## 6. Build the source_overt adjunct companion set

In [ ]:
!python -m src.analysis.build_c_source_overt_adjunct

## 7. Extract activations (stage-major, resumable)

In [ ]:
!python -m src.analysis.v2_pipeline extract --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt

## 8. CPU cross-check: all stages bound to the frozen benchmark

In [ ]:
!python -m src.analysis.verify_activations